Objetivo: realizar web scrapping para obtener dos libros

Librerias:
- pip install beautifulsoup4 fpdf weasyprint

In [14]:
from bs4 import BeautifulSoup
import requests
from fpdf import FPDF

In [ ]:

base_url = "https://basecamp.com/gettingreal"

# 1. Obtener lista de capítulos
response = requests.get(base_url)
soup = BeautifulSoup(response.text, "html.parser")


In [6]:
# Encontrar todos los enlaces a capítulos
chapter_links = []
for a in soup.select("a[href]"):
    href = a["href"]
    if href.startswith("/gettingreal/") and href != "/gettingreal/":
        chapter_links.append("https://basecamp.com" + href)


In [ ]:
# 2. preparar el PDF
pdf = FPDF()
pdf.set_auto_page_break(auto=True, margin=15)
pdf.set_font("Arial", size=12)

# 3. Recorrer capítulos y añadir al PDF
for link in chapter_links:
    res = requests.get(link)
    chapter_soup = BeautifulSoup(res.text, "html.parser")

    title = chapter_soup.find("h1").get_text(strip=True)
    content_elem = chapter_soup.select_one("main") or chapter_soup.select_one(".chapter")
    chapter_text = content_elem.get_text("\n", strip=True) if content_elem else ""

    pdf.add_page()
    pdf.set_font("Arial", "B", 16)
    pdf.multi_cell(0, 10, title)
    pdf.ln()

    pdf.set_font("Arial", size=12)
    pdf.multi_cell(0, 8, chapter_text)

# 4. Guardar el PDF
pdf.output("getting_real.pdf")
print("Libro guardado en getting_real.pdf")

RuntimeError: TTF Font file not found: DejaVuSans.ttf

In [16]:
import requests
from bs4 import BeautifulSoup
from weasyprint import HTML

base_url = "https://basecamp.com/gettingreal"

# 1. Obtener lista de capítulos
response = requests.get(base_url)
soup = BeautifulSoup(response.text, "html.parser")

chapter_links = []
for a in soup.select("a[href]"):
    href = a["href"]
    if href.startswith("/gettingreal/") and href != "/gettingreal/":
        chapter_links.append("https://basecamp.com" + href)

chapter_links = sorted(set(chapter_links))

# 2. Construir HTML del libro
html_content = "<h1>Getting Real - Basecamp</h1>"
for link in chapter_links:
    res = requests.get(link)
    chapter_soup = BeautifulSoup(res.text, "html.parser")

    title = chapter_soup.find("h1").get_text(strip=True)
    content_elem = chapter_soup.select_one("main") or chapter_soup.select_one(".chapter")
    chapter_text = content_elem.prettify() if content_elem else ""

    html_content += f"<h2>{title}</h2>{chapter_text}<hr>"

# 3. Generar PDF
HTML(string=html_content).write_pdf("getting_real.pdf")

print("Libro guardado en getting_real.pdf")


Libro guardado en getting_real.pdf


In [18]:
import requests
from bs4 import BeautifulSoup
from weasyprint import HTML

base_url = "https://basecamp.com/gettingreal"

# 1. Obtener lista de capítulos
response = requests.get(base_url)
response.encoding = "utf-8"  # Fuerza UTF-8
soup = BeautifulSoup(response.text, "html.parser")

chapter_links = []
for a in soup.select("a[href]"):
    href = a["href"]
    if href.startswith("/gettingreal/") and href != "/gettingreal/":
        chapter_links.append("https://basecamp.com" + href)

chapter_links = sorted(set(chapter_links))

# 2. Construir HTML del libro
html_content = """
<h1 style='text-align:center;'>Getting Real - Basecamp</h1>
<p style='text-align:center; font-size:14px;'>Libro recopilado automáticamente para uso personal</p>
<hr>
"""

for i, link in enumerate(chapter_links):
    res = requests.get(link)
    res.encoding = "utf-8"  # Asegura UTF-8
    chapter_soup = BeautifulSoup(res.text, "html.parser")

    # Título del capítulo
    title = chapter_soup.find("h1").get_text(strip=True)

    # Contenido principal
    content_elem = chapter_soup.select_one("main") or chapter_soup.select_one(".chapter")
    if not content_elem:
        continue

    # 3. Eliminar botones "Next Chapter" y textos finales no deseados
    for unwanted in content_elem.find_all(["a", "button"], recursive=True):
        unwanted.decompose()

    # Eliminar texto publicitario final
    for p in content_elem.find_all("p"):
        if "We made Basecamp" in p.get_text() or "Copyright" in p.get_text():
            p.decompose()

    # Convertir contenido limpio a HTML
    chapter_html = content_elem.prettify()

    # Evitar repetir índice/título principal en capítulos después del primero
    html_content += f"<h2>{title}</h2>{chapter_html}<hr>"

# 4. Generar PDF
HTML(string=html_content).write_pdf("getting_real_v2.pdf")
print("Libro guardado en getting_real.pdf")


Libro guardado en getting_real.pdf


In [18]:
!pip install weasyprint


   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.1 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.1 MB 1.3 MB/s eta 0:00:02
   --------------- ------------------------ 0.8/2.1 MB 1.5 MB/s eta 0:00:01
   ------------------------- -------------- 1.3/2.1 MB 1.8 MB/s eta 0:00:01
   ----------------------------------- ---- 1.8/2.1 MB 2.0 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 2.0 MB/s  0:00:01

   -------- ------------------------------- 2/9 [tinyhtml5]
   ------------- -------------------------- 3/9 [Pyphen]
   ------------- -------------------------- 3/9 [Pyphen]
   ---------------------- ----------------- 5/9 [pycparser]
   ---------------------- ----------------- 5/9 [pycparser]
   -------------------------- ------------- 6/9 [cssselect2]
   ------------------------------- -------- 7/9 [cffi]
   ------------------------------- -------- 7/9 [cffi]
   ----------------

In [ ]:
import requests
from bs4 import BeautifulSoup
from weasyprint import HTML

def scrapping2pdf(link, titulo, descripcion):
    nombre_pdf= link.rstrip("/").split("/")[-1]

    # 2. Construir HTML del libro con título principal
    html_content = """
    <h1 style='text-align:center;'>{titulo}</h1>
    <p style='text-align:center; font-size:14px;'>{descripcion}</p>
    <p style='text-align:center; font-size:14px;'>Libro recopilado de {link}</p>
    <hr>
    """
    html_content= html_content.format(titulo=titulo, descripcion= descripcion, link= link)

    # 1. Obtener lista de capítulos
    response = requests.get(link)
    response.encoding = "utf-8"
    soup = BeautifulSoup(response.text, "html.parser")

    chapter_links = []
    for a in soup.select("a[href]"):
        href = a["href"]
        if href.startswith(f"/{nombre_pdf}/") and href != f"/{nombre_pdf}/":
            chapter_links.append("https://basecamp.com" + href)

    chapter_links = sorted(set(chapter_links))

    # 3. Crear índice una sola vez
    html_content += "<h2>Index</h2><ol>"
    for i, link in enumerate(chapter_links, 1):
        res = requests.get(link)
        res.encoding = "utf-8"
        chapter_soup = BeautifulSoup(res.text, "html.parser")
        title = chapter_soup.find("h1").get_text(strip=True)
        html_content += f" Chapter {i}. {title}</p>"
    html_content += "</ol><hr>"

    # 4. Agregar capítulos
    for i, link in enumerate(chapter_links):
        res = requests.get(link)
        res.encoding = "utf-8" 
        chapter_soup = BeautifulSoup(res.text, "html.parser")

        # Título
        title = chapter_soup.find("h1").get_text(strip=True)

        # Contenido principal
        content_elem = chapter_soup.select_one("div.content")

        if not content_elem:
            continue

        # Eliminar solo footers
        for unwanted in content_elem.find_all(["footer"], recursive=True):
            unwanted.decompose()

        # Eliminar publicidad o copyright
        for p in content_elem.find_all("p"):
            if "We made" in p.get_text() or "Copyright" in p.get_text():
                p.decompose()

        # Mantener enlaces: si vacíos, poner href como texto
        for a in content_elem.find_all("a"):
            if not a.get_text(strip=True):
                a.string = a["href"]

        # Agregar capítulo
        chapter_html = content_elem.prettify()
        # html_content += f"<h2>{title}</h2>{chapter_html}<hr>"
        html_content += f"<h3>Chapter {i+1} </h3><hr>"
        html_content += f"<h1>{title}</h1>{chapter_html}<hr>"

    # 5. Generar PDF
    HTML(string=html_content).write_pdf(f"../doc_pdf/{nombre_pdf}.pdf")
    print(f"✅ Libro {nombre_pdf}.pdf extraido correctamente.")


Prueba correcta

In [5]:
from fpdf import FPDF

def scrapping2pdf_fpdf(link, titulo, descripcion):
    import requests
    from bs4 import BeautifulSoup

    nombre_pdf = link.rstrip("/").split("/")[-1]
    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    
    # Título
    pdf.add_page()
    pdf.set_font("Arial", 'B', 16)
    pdf.multi_cell(0, 10, titulo, align="C")
    
    pdf.set_font("Arial", '', 12)
    pdf.multi_cell(0, 8, descripcion, align="C")
    pdf.multi_cell(0, 8, f"Libro recopilado de {link}", align="C")
    pdf.ln(5)

    # Obtener capítulos
    response = requests.get(link)
    response.encoding = "utf-8"
    soup = BeautifulSoup(response.text, "html.parser")
    chapter_links = ["https://basecamp.com" + a["href"] for a in soup.select("a[href]") 
                     if a["href"].startswith(f"/{nombre_pdf}/") and a["href"] != f"/{nombre_pdf}/"]
    chapter_links = sorted(set(chapter_links))

    for i, chapter_url in enumerate(chapter_links):
        res = requests.get(chapter_url)
        res.encoding = "utf-8"
        chapter_soup = BeautifulSoup(res.text, "html.parser")
        title = chapter_soup.find("h1").get_text(strip=True)
        content_elem = chapter_soup.select_one("div.content")
        if not content_elem:
            continue
        text = content_elem.get_text("\n", strip=True)

        pdf.add_page()
        pdf.set_font("Arial", 'B', 14)
        pdf.multi_cell(0, 8, f"Chapter {i+1}: {title}")
        pdf.set_font("Arial", '', 12)
        pdf.multi_cell(0, 6, text)

    pdf.output(f"../doc_pdf/{nombre_pdf.title()}_Basecamp.pdf")
    print(f"✅ Libro {nombre_pdf}.pdf extraido correctamente.")


In [2]:
nomre= 'fsdfsdf'
nomre.title()

'Fsdfsdf'

In [6]:
# Realizamos el llamado a la funcion
link = "https://basecamp.com/gettingreal"
titulo= "Getting Real - Basecamp "
descripcion ="The smarter, faster, easier way to build a successful web application"

scrapping2pdf_fpdf(link, titulo, descripcion)

C:\Users\Angelica\AppData\Local\Temp\ipykernel_8840\3403927604.py:13: DeprecationWarning: Substituting font arial by core font helvetica - This is deprecated since v2.7.8, and will soon be removed
  pdf.set_font("Arial", 'B', 16)
C:\Users\Angelica\AppData\Local\Temp\ipykernel_8840\3403927604.py:16: DeprecationWarning: Substituting font arial by core font helvetica - This is deprecated since v2.7.8, and will soon be removed
  pdf.set_font("Arial", '', 12)


FPDFException: Not enough horizontal space to render a single character